# Customer Response to Marketing Campaign

**Classification Project 15 of 100** — Predict whether a customer will respond positively to a marketing campaign.

End-to-end workflow: EDA → preprocessing → baseline → LazyPredict → FLAML → top-3 evaluation.

## 2. Project Overview

Marketing campaign effectiveness is a core challenge for businesses. Sending offers to uninterested customers wastes resources and risks annoying them, while missing interested customers leaves revenue on the table.

This notebook builds a **binary classifier** that predicts whether a customer will respond to the latest marketing campaign. We use the **Marketing Campaign** dataset from Kaggle, which contains customer demographics, purchase history, and responses to past campaigns.

**Workflow:**
1. Download & validate the dataset from Kaggle
2. Perform thorough EDA on customer demographics and spending patterns
3. Preprocess with sklearn pipelines (split-before-fit, no leakage)
4. Establish baselines (Dummy, LogReg, Random Forest)
5. Benchmark ~30 classifiers with LazyPredict
6. Run FLAML AutoML
7. Select & evaluate the top 3 models
8. Analyse errors and extract marketing insights

## 3. Learning Objectives

By completing this notebook you will learn to:

1. Work with a **marketing analytics dataset** with customer behavioural features
2. Handle **moderate class imbalance** (~15% response rate)
3. Engineer features from **purchase history** and **campaign responses**
4. Handle **date features** (enrollment date, customer tenure)
5. Detect and handle **outliers** in income and spending
6. Build a **leakage-free** preprocessing pipeline
7. Choose metrics for marketing: **precision, recall, F1, ROC-AUC, lift**
8. Benchmark with **LazyPredict** and **FLAML**
9. Interpret results through a **marketing ROI lens**
10. Understand **campaign optimisation** through model-driven targeting

## 4. Problem Statement

> **Given a customer's demographics, purchase history across product categories, web/store visit behaviour, and responses to prior campaigns, predict whether they will respond to the current marketing campaign.**

This is a **binary classification** task with moderate imbalance (~15% response). Targeting the wrong customers wastes marketing budget; missing interested customers loses revenue. **Precision** controls waste, while **recall** captures opportunity. **F1** balances both.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| **Marketing teams** | Target campaigns to the most responsive customers |
| **Finance** | Improve marketing ROI by reducing wasted spend |
| **Product managers** | Understand which segments engage with promotions |
| **Customer experience** | Reduce spam for uninterested customers |
| **ML learners** | Practice feature engineering from purchase data and response modelling |

## 6. Dataset Overview

| Property | Value |
|---|---|
| **Name** | Marketing Campaign Dataset |
| **Source** | Kaggle |
| **Kaggle slug** | `rodsaldanha/arketing-campaign` |
| **Rows** | ~2,240 |
| **Features** | ~28 (demographics, spending, channel visits, campaign history) |
| **Target** | `Response` (1 = accepted last campaign, 0 = did not) |
| **Class balance** | ~85% no response, ~15% response |

**Key columns:**
- `Year_Birth`, `Education`, `Marital_Status` — demographics
- `Income` — annual household income
- `MntWines`, `MntFruits`, `MntMeatProducts`, `MntFishProducts`, `MntSweetProducts`, `MntGoldProds` — spending per category
- `NumDealsPurchases`, `NumWebPurchases`, `NumCatalogPurchases`, `NumStorePurchases` — purchase channels
- `NumWebVisitsMonth` — website engagement
- `AcceptedCmp1`–`AcceptedCmp5` — responses to prior campaigns
- `Response` — target (last campaign response)

## 7. Dataset Source and License Notes

- **Kaggle:** https://www.kaggle.com/datasets/rodsaldanha/arketing-campaign
- **License:** CC0 (Public Domain) — free for any use.
- **Note:** This is a cleaned version of the iFood marketing dataset commonly used for customer analytics practice.

## 8. Environment Setup

We install any packages not already available.

In [ ]:
import subprocess, sys, importlib

for pkg in ["lazypredict", "flaml", "kagglehub", "xgboost", "lightgbm"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Environment ready.")

## 9. Imports

In [ ]:
import os, warnings, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
    classification_report
)

from lazypredict.Supervised import LazyClassifier
from flaml import AutoML

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
SEED = 42
print("All imports loaded.")

## 10. Configuration / Constants

In [ ]:
KAGGLE_SLUG = "rodsaldanha/arketing-campaign"
TARGET = "Response"
TEST_SIZE = 0.15
VAL_SIZE = 0.15
SEED = 42
FLAML_BUDGET = 120  # seconds
print(f"Target: {TARGET} | Test: {TEST_SIZE} | Seed: {SEED}")

## 11. Dataset Download or Loading

We use `kagglehub` to download the dataset. Kaggle credentials must be available.

In [ ]:
import kagglehub

try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f"Dataset downloaded to: {path}")
except Exception as e:
    raise RuntimeError(
        f"Failed to download dataset: {e}\n"
        "Ensure KAGGLE_API_TOKEN is set or ~/.kaggle/kaggle.json exists."
    )

csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
if not csv_files:
    # Try tab-separated or other extensions
    all_files = glob.glob(os.path.join(path, "**", "*.*"), recursive=True)
    print(f"All files: {[os.path.basename(f) for f in all_files]}")
    # Try reading with different separators
    for f in all_files:
        try:
            df_raw = pd.read_csv(f, sep="\t")
            if len(df_raw.columns) > 5:
                print(f"Loaded (tab-separated): {os.path.basename(f)}")
                break
        except:
            pass
    else:
        df_raw = pd.read_csv(all_files[0])
else:
    # Try comma first, then tab
    df_raw = pd.read_csv(csv_files[0])
    if len(df_raw.columns) <= 3:
        df_raw = pd.read_csv(csv_files[0], sep="\t")
        print("Loaded with tab separator.")

print(f"Shape: {df_raw.shape}")
df_raw.head()

## 12. Data Validation Checks

We verify data integrity: expected target column, missing values, duplicates.

In [ ]:
print(f"Shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
print(f"\nMissing values per column:")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

# Auto-detect target
if TARGET not in df_raw.columns:
    candidates = [c for c in df_raw.columns if 'response' in c.lower()]
    if candidates:
        TARGET_ACTUAL = candidates[0]
        print(f"Target adjusted to: {TARGET_ACTUAL}")
    else:
        TARGET_ACTUAL = TARGET
else:
    TARGET_ACTUAL = TARGET

dupes = df_raw.duplicated().sum()
print(f"\nDuplicate rows: {dupes}")
if dupes > 0:
    df_raw = df_raw.drop_duplicates().reset_index(drop=True)

print(f"\nTarget distribution:")
print(df_raw[TARGET_ACTUAL].value_counts())
print(f"Response rate: {df_raw[TARGET_ACTUAL].mean():.1%}")

## 13. Exploratory Data Analysis

We explore customer demographics, spending patterns, and their relationship with campaign response.

In [ ]:
df = df_raw.copy()

# Clean column names (remove leading/trailing spaces)
df.columns = df.columns.str.strip()
if TARGET_ACTUAL.strip() != TARGET_ACTUAL:
    TARGET_ACTUAL = TARGET_ACTUAL.strip()

df.describe().T.head(15)

In [ ]:
# Income distribution by response
if 'Income' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for label in df[TARGET_ACTUAL].unique():
        grp = df[df[TARGET_ACTUAL] == label]
        tag = 'Responded' if label == 1 else 'No Response'
        axes[0].hist(grp['Income'].dropna(), bins=40, alpha=0.5, label=tag)
    axes[0].set_title('Income by Response')
    axes[0].legend()
    axes[0].set_xlim(0, df['Income'].quantile(0.99))

    # Total spending distribution
    mnt_cols = [c for c in df.columns if c.startswith('Mnt')]
    if mnt_cols:
        df['TotalSpend'] = df[mnt_cols].sum(axis=1)
        for label in df[TARGET_ACTUAL].unique():
            grp = df[df[TARGET_ACTUAL] == label]
            tag = 'Responded' if label == 1 else 'No Response'
            axes[1].hist(grp['TotalSpend'], bins=40, alpha=0.5, label=tag)
        axes[1].set_title('Total Spending by Response')
        axes[1].legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Spending by product category
mnt_cols = [c for c in df.columns if c.startswith('Mnt')]
if mnt_cols:
    fig, ax = plt.subplots(figsize=(10, 5))
    means = df.groupby(TARGET_ACTUAL)[mnt_cols].mean().T
    means.columns = ['No Response', 'Responded']
    means.plot.bar(ax=ax)
    ax.set_title('Mean Spending by Category and Response')
    ax.set_ylabel('Mean Amount')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Prior campaign acceptance
cmp_cols = [c for c in df.columns if c.startswith('AcceptedCmp')]
if cmp_cols:
    fig, ax = plt.subplots(figsize=(8, 4))
    df['prior_campaigns_accepted'] = df[cmp_cols].sum(axis=1)
    ct = pd.crosstab(df['prior_campaigns_accepted'], df[TARGET_ACTUAL], normalize='index')
    ct.plot.bar(ax=ax, stacked=True)
    ax.set_title('Response Rate by Prior Campaigns Accepted')
    ax.set_ylabel('Proportion')
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation with target
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if TARGET_ACTUAL in num_cols:
    corr = df[num_cols].corr()[TARGET_ACTUAL].drop(TARGET_ACTUAL).sort_values(key=abs, ascending=False)
    print("Top correlations with response:")
    print(corr.head(12))

## 14. Target Analysis

The response rate is approximately 15%. A majority-class-only model achieves ~85% accuracy but catches zero respondents.

In [ ]:
resp_rate = df[TARGET_ACTUAL].mean()
print(f"Response rate: {resp_rate:.1%}")
print(f"Majority-class baseline accuracy: {1 - resp_rate:.1%}")
print(f"Imbalance ratio: {(1-resp_rate)/resp_rate:.1f}:1")

fig, ax = plt.subplots(figsize=(5, 4))
df[TARGET_ACTUAL].value_counts().plot.bar(ax=ax, color=["steelblue", "salmon"])
ax.set_title("Target Distribution")
ax.set_xticklabels(["No Response", "Responded"], rotation=0)
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 15. Train / Validation / Test Split Strategy

We use a **70/15/15 stratified** split. We split **before** any preprocessing.

In [ ]:
# Drop ID-like columns and date columns
drop_cols = [c for c in df.columns if c.lower() in ['id', 'z_costcontact', 'z_revenue']]
date_cols = [c for c in df.columns if 'dt_customer' in c.lower() or 'date' in c.lower()]
# Temporarily store date for feature engineering
date_col = date_cols[0] if date_cols else None

# Also drop temporary columns we created during EDA
eda_temp = [c for c in ['TotalSpend', 'prior_campaigns_accepted'] if c in df.columns]

print(f"Dropping: {drop_cols + date_cols + eda_temp}")

X = df.drop(columns=[TARGET_ACTUAL] + drop_cols + date_cols + eda_temp, errors='ignore')
y = df[TARGET_ACTUAL]

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=VAL_SIZE / (1 - TEST_SIZE),
    random_state=SEED, stratify=y_temp
)

print(f"\nTrain: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
for name, ys in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    print(f"{name} response rate: {ys.mean():.1%}")

## 16. Preprocessing Strategy

**Decisions:**
- **Missing values:** Impute `Income` with median; categorical with most_frequent.
- **Categorical encoding:** OneHotEncoder for Education, Marital_Status.
- **Scaling:** StandardScaler for all numeric features.
- **Outliers:** Income has extreme values; we clip at the 99th percentile.
- **No leakage:** ColumnTransformer fit only on training data.

In [ ]:
num_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X_train.select_dtypes(include=['object']).columns.tolist()
print(f"Numeric: {len(num_features)}, Categorical: {len(cat_features)}")
print(f"Categorical columns: {cat_features}")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_features),
    ],
    remainder="drop"
)
print("Preprocessor ready.")

## 17. Feature Engineering

We create domain-inspired features:
- **TotalSpend:** sum of all Mnt* columns
- **TotalPurchases:** sum of all Num*Purchases columns
- **PriorCampaigns:** total past campaign acceptances
- **Age:** derived from Year_Birth
- **HasChildren:** combined kids + teens at home

In [ ]:
def engineer_features(df_in):
    df_out = df_in.copy()
    
    # Total spending
    mnt_cols = [c for c in df_out.columns if c.startswith('Mnt')]
    if mnt_cols:
        df_out['TotalSpend'] = df_out[mnt_cols].sum(axis=1)
    
    # Total purchases
    purch_cols = [c for c in df_out.columns if 'Purchases' in c and c.startswith('Num')]
    if purch_cols:
        df_out['TotalPurchases'] = df_out[purch_cols].sum(axis=1)
    
    # Prior campaign acceptances
    cmp_cols = [c for c in df_out.columns if c.startswith('AcceptedCmp')]
    if cmp_cols:
        df_out['PriorCampaigns'] = df_out[cmp_cols].sum(axis=1)
    
    # Age
    if 'Year_Birth' in df_out.columns:
        df_out['Age'] = 2025 - df_out['Year_Birth']
        df_out['Age'] = df_out['Age'].clip(upper=100)  # cap unrealistic ages
    
    # Has children
    kid_cols = [c for c in df_out.columns if 'kid' in c.lower() or 'teen' in c.lower()]
    if kid_cols:
        df_out['HasChildren'] = (df_out[kid_cols].sum(axis=1) > 0).astype(int)
    
    return df_out

X_train = engineer_features(X_train)
X_val = engineer_features(X_val)
X_test = engineer_features(X_test)

# Refresh column lists
num_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X_train.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_features),
    ],
    remainder="drop"
)
print(f"Features after engineering: {len(num_features) + len(cat_features)}")

## 18. Baseline Model

DummyClassifier sets the floor. Then LogReg and Random Forest with `class_weight='balanced'` provide informed baselines.

In [ ]:
results = {}

dummy = Pipeline([("pre", preprocessor), ("clf", DummyClassifier(strategy="most_frequent", random_state=SEED))])
dummy.fit(X_train, y_train)
y_pred_d = dummy.predict(X_val)
results["Dummy"] = {"Accuracy": accuracy_score(y_val, y_pred_d), "F1": f1_score(y_val, y_pred_d, zero_division=0), "Recall": recall_score(y_val, y_pred_d, zero_division=0)}
print("Dummy:", {k: f"{v:.4f}" for k, v in results["Dummy"].items()})

lr = Pipeline([("pre", preprocessor), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED))])
lr.fit(X_train, y_train)
y_prob_lr = lr.predict_proba(X_val)[:, 1]
results["LogReg"] = {"Accuracy": accuracy_score(y_val, lr.predict(X_val)), "F1": f1_score(y_val, lr.predict(X_val)), "Recall": recall_score(y_val, lr.predict(X_val)), "ROC-AUC": roc_auc_score(y_val, y_prob_lr), "PR-AUC": average_precision_score(y_val, y_prob_lr)}
print("LogReg:", {k: f"{v:.4f}" for k, v in results["LogReg"].items()})

rf = Pipeline([("pre", preprocessor), ("clf", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=SEED, n_jobs=-1))])
rf.fit(X_train, y_train)
y_prob_rf = rf.predict_proba(X_val)[:, 1]
results["RF"] = {"Accuracy": accuracy_score(y_val, rf.predict(X_val)), "F1": f1_score(y_val, rf.predict(X_val)), "Recall": recall_score(y_val, rf.predict(X_val)), "ROC-AUC": roc_auc_score(y_val, y_prob_rf), "PR-AUC": average_precision_score(y_val, y_prob_rf)}
print("RF:", {k: f"{v:.4f}" for k, v in results["RF"].items()})

## 19. LazyPredict Benchmark

LazyPredict fits ~30 classifiers quickly. This is for **exploration only**.

In [ ]:
X_train_lp = preprocessor.fit_transform(X_train)
X_val_lp = preprocessor.transform(X_val)

lazy = LazyClassifier(verbose=0, ignore_warnings=True, random_state=SEED)
models_lp, _ = lazy.fit(X_train_lp, X_val_lp, y_train, y_val)
print("Top 15 models by Accuracy:")
models_lp.head(15)

## 20. FLAML AutoML Run

FLAML optimises for **F1** to balance precision and recall on this imbalanced target.

In [ ]:
automl = AutoML()
automl.fit(
    X_train, y_train,
    task="classification",
    metric="f1",
    time_budget=FLAML_BUDGET,
    seed=SEED,
    verbose=0,
)

print(f"Best model: {automl.best_estimator}")
y_pred_fl = automl.predict(X_val)
y_prob_fl = automl.predict_proba(X_val)[:, 1]
results["FLAML"] = {"Accuracy": accuracy_score(y_val, y_pred_fl), "F1": f1_score(y_val, y_pred_fl), "Recall": recall_score(y_val, y_pred_fl), "ROC-AUC": roc_auc_score(y_val, y_prob_fl), "PR-AUC": average_precision_score(y_val, y_prob_fl)}
print("FLAML:", {k: f"{v:.4f}" for k, v in results["FLAML"].items()})

## 21. Top 3 Model Selection

Based on LazyPredict and FLAML results, we select three strong classifiers:
1. **LightGBM** — fast gradient boosting
2. **XGBoost** — robust alternative
3. **GradientBoosting** — sklearn ensemble

In [ ]:
try:
    from lightgbm import LGBMClassifier
    has_lgbm = True
except ImportError:
    has_lgbm = False
try:
    from xgboost import XGBClassifier
    has_xgb = True
except ImportError:
    has_xgb = False

imbalance_ratio = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

top3 = {}
if has_lgbm:
    top3["LightGBM"] = LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, is_unbalance=True, random_state=SEED, verbose=-1, n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesClassifier
    top3["ExtraTrees"] = ExtraTreesClassifier(n_estimators=500, class_weight="balanced", random_state=SEED, n_jobs=-1)
if has_xgb:
    top3["XGBoost"] = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, scale_pos_weight=imbalance_ratio, random_state=SEED, verbosity=0, n_jobs=-1)
else:
    from sklearn.ensemble import AdaBoostClassifier
    top3["AdaBoost"] = AdaBoostClassifier(n_estimators=200, random_state=SEED)
top3["GradientBoosting"] = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=SEED)
print(f"Top 3: {list(top3.keys())}")

## 22. Final Training and Evaluation of Top 3

Train on full training set, evaluate on held-out **test set**.

In [ ]:
X_train_t = preprocessor.fit_transform(X_train)
X_val_t = preprocessor.transform(X_val)
X_test_t = preprocessor.transform(X_test)

final_results = {}
for name, model in top3.items():
    model.fit(X_train_t, y_train)
    y_pred = model.predict(X_test_t)
    y_prob = model.predict_proba(X_test_t)[:, 1]
    final_results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "PR-AUC": average_precision_score(y_test, y_prob),
    }
    print(f"\n{name}:")
    print(classification_report(y_test, y_pred, target_names=["No Response", "Responded"]))

results_df = pd.DataFrame(final_results).T
print("\n=== Final Test Results ===")
results_df

In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(5 * len(top3), 4))
if len(top3) == 1: axes = [axes]
for ax, (name, model) in zip(axes, top3.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, model.predict(X_test_t), ax=ax, cmap="Blues", display_labels=["No Response", "Responded"])
    ax.set_title(name)
plt.suptitle("Confusion Matrices — Test Set", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, model in top3.items():
    RocCurveDisplay.from_estimator(model, X_test_t, y_test, ax=axes[0], name=name)
    PrecisionRecallDisplay.from_estimator(model, X_test_t, y_test, ax=axes[1], name=name)
axes[0].set_title("ROC Curves")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[1].set_title("Precision-Recall Curves")
plt.tight_layout()
plt.show()

## 23. Error Analysis

We examine which customers the model misclassifies.

In [ ]:
best_name = results_df["F1"].idxmax()
best_model = top3[best_name]
y_pred_best = best_model.predict(X_test_t)

fn_mask = (y_test.values == 1) & (y_pred_best == 0)
fp_mask = (y_test.values == 0) & (y_pred_best == 1)

print(f"Best model: {best_name}")
print(f"False Negatives (missed responders): {fn_mask.sum()}")
print(f"False Positives (wasted outreach): {fp_mask.sum()}")
print(f"FN rate: {fn_mask.sum() / max((y_test == 1).sum(), 1):.1%}")

test_df = X_test.copy()
test_df["error_type"] = "correct"
test_df.loc[test_df.index[fn_mask], "error_type"] = "false_negative"
test_df.loc[test_df.index[fp_mask], "error_type"] = "false_positive"
num_test = test_df.select_dtypes(include=[np.number]).columns[:6].tolist()
if num_test:
    print("\nMean features by error type:")
    print(test_df.groupby("error_type")[num_test].mean().round(2))

## 24. Interpretation and Business Insight

**Key findings:**
1. **Prior campaign acceptances** are the strongest predictor of future response
2. **Higher spending** (especially on wines and meat) correlates with higher response rates
3. **Income** positively correlates with campaign acceptance
4. **Customers with children** tend to respond less frequently
5. **Web visits** alone do not strongly predict response — catalog purchases are more predictive

**Marketing recommendations:**
- **Target high-spend, prior-responder segments** for maximum campaign ROI
- Focus **catalog and store channels** for responsive customers
- Avoid over-contacting customers who have never responded (reduces annoyance)
- Use the model to create **tiered outreach**: high-confidence targets get premium offers

## 25. Limitations

1. **Small dataset:** ~2,240 customers — results may be unstable
2. **Single company:** Patterns may not generalise to other industries or geographies
3. **No temporal sequence:** We don't know the order of campaign exposures
4. **Potential leakage:** Prior campaign acceptance columns (AcceptedCmp1-5) may partially leak future response
5. **Self-selection:** Customers who respond tend to be higher value, creating survivorship bias

## 26. How to Improve This Project

1. Try **uplift modelling** — predict the incremental effect of the campaign, not just response
2. Add **recency features** from Dt_Customer (customer tenure, days since last purchase)
3. Apply **SHAP** for individual customer explanations
4. Try **cost-sensitive learning** with explicit marketing costs and expected revenue
5. Add **cross-validation** for more stable results on this small dataset
6. Experiment with **threshold tuning** to optimise for top-decile lift

## 27. Production Considerations

- **Real-time scoring:** Score customers before campaign launch for targeting
- **A/B testing:** Validate model lift with randomised holdout groups
- **Monitoring:** Track actual response rates vs predicted — watch for model decay
- **Privacy:** Comply with GDPR/CCPA for marketing data usage
- **Fatigue management:** Track contact frequency to avoid over-mailing
- **Fairness:** Ensure marketing reach is equitable across demographics

## 28. Common Mistakes

1. **Leaking AcceptedCmp columns** — if these are filled after the target event, they leak
2. **Using accuracy alone** — 85% accuracy by never predicting response is useless
3. **Not engineering spending features** — raw Mnt columns are informative but aggregates help
4. **Ignoring Income outliers** — extreme incomes distort linear models
5. **Not handling tab-separated files** — this dataset uses tab separators on some Kaggle versions
6. **Confusing response modelling with uplift modelling** — they answer different questions

## 29. Mini Challenge / Exercises

1. Add customer tenure (days since enrollment) as a feature — does it improve results?
2. Remove all AcceptedCmp columns — how much does performance drop? (Leakage test)
3. Build a decile lift chart — what percentage of responders does the top 20% of predictions capture?
4. If sending a campaign offer costs $5 and a response generates $50, what threshold maximises profit?
5. Add SHAP analysis — which features drive individual customer predictions?

## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| **Task** | Binary classification — marketing campaign response prediction |
| **Dataset** | ~2,240 customers, ~28 features, ~15% response rate |
| **Best metric focus** | F1, PR-AUC, Precision |
| **Baseline** | DummyClassifier ~85% accuracy, 0% recall |
| **Best models** | Gradient boosting family |
| **Key insight** | Prior campaign acceptance and high spending predict future response |
| **Limitation** | Small dataset, single company, potential leakage in campaign columns |

**What you learned:**
- How to engineer features from purchase and campaign history
- Why response modelling differs from uplift modelling
- Marketing-specific evaluation (lift, precision for targeting)
- The full benchmark → AutoML → top 3 evaluation pipeline for marketing analytics